In [ ]:
from pathlib import Path
import shutil

import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

In [ ]:
RAW_DATA_DIR = Path("raw_data")
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

RAW_FILES = {
    "202602": [RAW_DATA_DIR / "stock_tick_202602.parquet"],
    "202603": [
        RAW_DATA_DIR / "stock_tick_20260302_0303.parquet",
        RAW_DATA_DIR / "stock_tick_202603.parquet",
    ],
    "202604": [RAW_DATA_DIR / "stock_tick_202604.parquet"],
}

SORT_COLUMNS = ["symbol", "tradingdate", "Time"]

In [ ]:
def copy_single_month(month: str) -> Path:
    """Save a single raw month to data/{month}.parquet."""
    src = RAW_FILES[month][0]
    dst = DATA_DIR / f"{month}.parquet"
    shutil.copy2(src, dst)
    return dst


def merge_sort_month_by_symbol_prefix(month: str, prefix_len: int = 4, batch_size: int = 1_000_000) -> Path:
    """Merge raw parquet files and write one file sorted by symbol, tradingdate, Time.

    The March data is too large for a comfortable one-shot pandas sort, so this
    does an external sort: stream rows into symbol-prefix partitions, then sort
    and append each partition in lexicographic prefix order.
    """
    sources = RAW_FILES[month]
    output_path = DATA_DIR / f"{month}.parquet"
    temp_dir = DATA_DIR / f"_tmp_{month}_partitions"

    if temp_dir.exists():
        shutil.rmtree(temp_dir)
    temp_dir.mkdir(parents=True)
    if output_path.exists():
        output_path.unlink()

    partition_counts = {}

    for src in sources:
        parquet_file = pq.ParquetFile(src)
        print(f"partitioning {src} ({parquet_file.metadata.num_rows:,} rows)")

        for batch_no, batch in enumerate(parquet_file.iter_batches(batch_size=batch_size), start=1):
            table = pa.Table.from_batches([batch])
            prefixes = pc.utf8_slice_codeunits(table["symbol"], 0, prefix_len)

            for prefix in sorted(prefixes.unique().to_pylist()):
                mask = pc.equal(prefixes, pa.scalar(prefix))
                part = table.filter(mask)
                prefix_dir = temp_dir / f"symbol_prefix={prefix}"
                prefix_dir.mkdir(exist_ok=True)

                part_no = partition_counts.get(prefix, 0)
                partition_counts[prefix] = part_no + 1
                pq.write_table(
                    part,
                    prefix_dir / f"part-{part_no:06d}.parquet",
                    compression="snappy",
                )

            print(f"  batch {batch_no}: wrote {table.num_rows:,} rows")

    output_writer = None
    try:
        for prefix_dir in sorted(p for p in temp_dir.iterdir() if p.is_dir()):
            print(f"sorting {prefix_dir.name}")
            part_df = pd.read_parquet(prefix_dir).sort_values(SORT_COLUMNS, kind="mergesort")
            part_table = pa.Table.from_pandas(part_df, preserve_index=False)

            if output_writer is None:
                output_writer = pq.ParquetWriter(output_path, schema=part_table.schema, compression="snappy")
            output_writer.write_table(part_table)
            del part_df, part_table
    finally:
        if output_writer is not None:
            output_writer.close()
        shutil.rmtree(temp_dir)

    return output_path

In [ ]:
# February and April each have one raw file, so save them under the target names.
for month in ["202602", "202604"]:
    saved_path = copy_single_month(month)
    print(f"saved {saved_path}")

# March has two raw files; merge and sort by symbol, tradingdate, Time.
saved_path = merge_sort_month_by_symbol_prefix("202603")
print(f"saved {saved_path}")

In [ ]:
# Basic check: output files exist and row counts match the raw files.
for month, sources in RAW_FILES.items():
    output_path = DATA_DIR / f"{month}.parquet"
    output_rows = pq.ParquetFile(output_path).metadata.num_rows
    raw_rows = sum(pq.ParquetFile(src).metadata.num_rows for src in sources)
    print(month, output_path, "rows:", output_rows, "expected:", raw_rows)